# NT Autofix: Greek cleanup + Latvian enforcement + Exact count matching

In-place edits on chapter JSONs (like OT `4split_json_upgraded`):

1. **Strip Greek punctuation** — apply `raccs_common` to every `greek` field  
2. **Greek count enforcement** — ensure JSON has exactly the same number of Greek words as strongs (stripped diacritics comparison)  
3. **Fix latvian junk** — remove entries that are just punctuation (`,` etc.)  
4. **Latvian count enforcement** — ensure JSON has exactly the same Latvian words as lv65 (stripped diacritics comparison)  
5. **Remove extras** — first from leftovers, then from mappings  
6. **Add missing** — missing latvian words added to leftover_latvian

## 0 – Config & shared helpers

In [1]:
from pathlib import Path
import json, re, unicodedata, os
from collections import Counter, defaultdict
import pandas as pd

BASE_DIR = Path("bible")
DRY_RUN  = False#True        # True = preview only, False = write files
VERBOSE  = False #True

# ── raccs_common: shared punctuation stripper ──────────────────────────────
def raccs_common(text):
    d = {
        ord('\u2019'): None,  # RIGHT SINGLE QUOTATION MARK  \u2019
        ord('\u2018'): None,  # LEFT SINGLE QUOTATION MARK   \u2018
        ord('\u201C'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('\u201D'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,
        ord(']'): None,
        ord('-'): None,
        ord("'"): None,
        ord('\u29FC'): None,  # \u29fc
        ord('\u29FD'): None,  # \u29fd
        ord('*'): None,
        ord('\u21D4'): None,  # \u21d4
        ord('\u3009'): None,  # \u3009
        ord('\u3008'): None,  # \u3008
        ord('\u203F'): None,  # \u203f
        ord('\u00AB'): None,  # \u00ab
        ord('\u00BB'): None,  # \u00bb
        ord('\u2039'): None,  # \u2039
        ord('\u203A'): None,  # \u203a
        ord('('): None,
        ord(')'): None,
        ord(';'): None,
        ord('{'): None,
        ord('}'): None,
    }
    return unicodedata.normalize('NFC', text).translate(d)

# ── Greek / Latvian helpers ───────────────────────────────────────────────
def strip_greek_diacritics(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

LV_CHARS = "A-Za-z\u0101\u0113\u012b\u016b\u0123\u0137\u013c\u0146\u0161\u017e\u010d\u0100\u0112\u012a\u016a\u0122\u0136\u013b\u0145\u0160\u017d\u010c"
LV_REGEX = re.compile(f"[{LV_CHARS}]+")

def lv_tokens(text: str):
    return LV_REGEX.findall(unicodedata.normalize('NFC', text))

def strip_lv_diacritics(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s.lower())
                   if unicodedata.category(c) != 'Mn')

## 1 – Load reference datasets

In [2]:
strongs_df = pd.read_csv("strongs.csv")
lv65_df    = pd.read_csv("latvian_full65.csv")

# Pre-clean strongs forms with raccs_common so we compare apples to apples
strongs_df["form_clean"] = strongs_df["form"].astype(str).apply(raccs_common)

strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
lv_g      = lv65_df.groupby(["book", "chapter", "verse"])

print(f"Strongs: {len(strongs_df)} rows, LV65: {len(lv65_df)} rows")

Strongs: 138993 rows, LV65: 7956 rows


## 2 – Dense compact writer (same as OT pattern)

In [3]:
def write_dense_json(file_path, data):
    """
    Write chapter data back in OT-style dense compact format.
    data = list of verse dicts, each with greek_words + leftover_latvian.
    """
    def word_to_compact(w):
        gr  = json.dumps(w.get('greek', ''), ensure_ascii=False)
        lv  = json.dumps(w.get('latvian', []), ensure_ascii=False)
        return f'{{ "greek": {gr}, "latvian": {lv} }}'

    verse_blocks = []
    for verse in data:
        words = verse.get('greek_words', [])
        compact_words = ',\n      '.join(word_to_compact(w) for w in words)
        lo_lv = json.dumps(verse.get('leftover_latvian', []), ensure_ascii=False)
        verse_blocks.append(
            f'  {{\n    "greek_words": [\n      {compact_words}\n    ],\n'
            f'    "leftover_latvian": {lo_lv}\n  }}'
        )

    new_raw = '[\n' + ',\n'.join(verse_blocks) + '\n]'
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(new_raw)

## 3 – Autofix function

For each verse in a chapter JSON:

**Pass 1 – Greek cleanup + count enforcement**  
**Pass 2 – Latvian junk removal**  
**Pass 3 – Latvian count enforcement:**  
- Match using stripped diacritics to avoid reruns  
- **Prefix fan-out**: `ne-` + `viens` + `var` → matches ref `neviens` + `nevar`  
  (one prefix entry can compound with multiple roots)  
- **Suffix merge**: `root` + `-suffix` → compound  
- Add missing ref words to leftover_latvian  
- Remove extras: first from leftovers, then from mappings

In [4]:
def autofix_chapter(book_name, chapter_num, strongs_g, lv_g, verbose=True, dry_run=True):
    """
    In-place autofix for a single NT chapter JSON.
    Returns (had_fixes: bool, fix_log: list).

    Ensures:
    - Greek word count matches strongs dataset exactly (stripped-diacritics comparison)
    - Latvian word count matches lv65 dataset exactly (stripped-diacritics comparison)
    - Handles prefix splits: "ne-" fans out to multiple roots
      e.g. "ne-" + "viens" + "var" → matches ref "neviens" + "nevar"
    - Extras removed: first from leftovers, then from mappings
    """
    json_path = BASE_DIR / book_name / f"{chapter_num}.json"
    if not json_path.exists():
        return False, []

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    fixes = []

    for vi, verse_data in enumerate(data):
        verse_num = vi + 1
        key = (book_name, chapter_num, verse_num)
        greek_words = verse_data.get('greek_words', [])
        leftover_latvian = verse_data.get('leftover_latvian', [])

        # ═══════════════════════════════════════════════════════════════════
        # PASS 1: Greek punctuation cleanup via raccs_common
        # ═══════════════════════════════════════════════════════════════════
        for wi, gw in enumerate(greek_words):
            old_gr = gw.get('greek', '')
            new_gr = raccs_common(old_gr).strip()
            if new_gr != old_gr:
                if verbose:
                    print(f"  ✏️🇬🇷 v{verse_num} w{wi+1}: '{old_gr}' → '{new_gr}'")
                greek_words[wi]['greek'] = new_gr
                fixes.append(('greek_clean', key, wi+1, old_gr, new_gr))

        # ═══════════════════════════════════════════════════════════════════
        # PASS 1b: Ensure Greek word count matches strongs exactly
        # ═══════════════════════════════════════════════════════════════════
        if key in strongs_g.groups:
            ref_strongs = strongs_g.get_group(key).sort_values('word')
            ref_count = len(ref_strongs)
            json_count = len(greek_words)

            if json_count > ref_count:
                ref_stripped = [strip_greek_diacritics(raccs_common(str(ref_strongs.iloc[i]['form']))).lower()
                                for i in range(ref_count)]
                json_stripped = [strip_greek_diacritics(gw.get('greek', '')).lower()
                                 for gw in greek_words]

                ref_matched = [False] * ref_count
                json_matched = [False] * json_count
                ri = 0
                for ji in range(json_count):
                    if ri < ref_count and json_stripped[ji] == ref_stripped[ri]:
                        json_matched[ji] = True
                        ref_matched[ri] = True
                        ri += 1

                if all(ref_matched):
                    extras = [i for i in range(json_count) if not json_matched[i]]
                    if extras:
                        for ei in reversed(extras):
                            removed = greek_words.pop(ei)
                            if verbose:
                                print(f"  ➖🇬🇷 v{verse_num} w{ei+1}: removed extra '{removed.get('greek', '')}' lv={removed.get('latvian', [])}")
                            salvaged_lv = removed.get('latvian', [])
                            if salvaged_lv:
                                leftover_latvian.extend(salvaged_lv)
                        fixes.append(('greek_extra_removed', key, len(extras)))

        # ═══════════════════════════════════════════════════════════════════
        # PASS 2: Latvian junk removal (pure punctuation entries)
        # ═══════════════════════════════════════════════════════════════════
        for wi, gw in enumerate(greek_words):
            lv_list = gw.get('latvian', [])
            cleaned = [w for w in lv_list if lv_tokens(w)]
            if len(cleaned) != len(lv_list):
                removed_junk = [w for w in lv_list if not lv_tokens(w)]
                if verbose:
                    print(f"  🗑️🇱🇻 v{verse_num} w{wi+1}: removed junk latvian {removed_junk}")
                greek_words[wi]['latvian'] = cleaned
                fixes.append(('lv_junk', key, wi+1, removed_junk))

        lo_cleaned = [w for w in leftover_latvian if lv_tokens(w)]
        if len(lo_cleaned) != len(leftover_latvian):
            removed_junk = [w for w in leftover_latvian if not lv_tokens(w)]
            if verbose:
                print(f"  🗑️🇱🇻 v{verse_num} leftover: removed junk {removed_junk}")
            leftover_latvian = lo_cleaned
            fixes.append(('lv_leftover_junk', key, removed_junk))

        # ═══════════════════════════════════════════════════════════════════
        # PASS 3: Latvian count enforcement — exact match with lv65
        #
        # Handles prefix/suffix splits in the JSON mappings:
        #   "ne-" is a prefix marker → fans out to ALL subsequent roots
        #     e.g. ne- + viens + var → matches ref "neviens" + "nevar"
        #   "-root" is a suffix marker → combines with preceding root
        #
        # Uses stripped diacritics for all comparisons.
        # ═══════════════════════════════════════════════════════════════════
        if key in lv_g.groups:
            lv_text = lv_g.get_group(key).iloc[0]['text']
            ref_lv = lv_tokens(lv_text)

            if ref_lv:
                # --- 3a. Collect raw entries from JSON ---
                all_raw_entries = []
                for gwi, gw in enumerate(greek_words):
                    for li, item in enumerate(gw.get('latvian', [])):
                        if item and item != '-':
                            all_raw_entries.append((item, 'mapped', gwi, li))
                for li, item in enumerate(leftover_latvian):
                    all_raw_entries.append((item, 'leftover', -1, li))

                # --- 3b. Tokenize and detect prefix/suffix entries ---
                json_tokens = []  # (stripped, original, entry_idx)
                for ei, (raw, src, gwi, li) in enumerate(all_raw_entries):
                    for tok in lv_tokens(unicodedata.normalize('NFC', raw)):
                        json_tokens.append((strip_lv_diacritics(tok), tok, ei))

                prefix_raw_indices = set()
                suffix_raw_indices = set()
                for ei, (raw, src, gwi, li) in enumerate(all_raw_entries):
                    stripped_raw = raw.strip()
                    if stripped_raw.endswith('-') and len(lv_tokens(stripped_raw)) == 1:
                        prefix_raw_indices.add(ei)
                    if stripped_raw.startswith('-') and len(lv_tokens(stripped_raw)) >= 1:
                        suffix_raw_indices.add(ei)

                # --- 3c. Count ref words by stripped diacritics ---
                ref_stripped_counts = Counter()
                ref_stripped_to_form = {}
                for w in ref_lv:
                    sw = strip_lv_diacritics(w)
                    ref_stripped_counts[sw] += 1
                    if sw not in ref_stripped_to_form:
                        ref_stripped_to_form[sw] = w

                # --- 3d. Match JSON tokens against ref ---
                ref_pool = Counter(ref_stripped_counts)
                # [stripped, original, entry_idx, matched]
                json_token_list = [[sw, orig, ei, False] for sw, orig, ei in json_tokens]

                # Phase 1: Direct matches (non-prefix, non-suffix tokens first)
                for jt in json_token_list:
                    if jt[3]:
                        continue
                    ei = jt[2]
                    if ei in prefix_raw_indices:
                        continue  # skip prefix tokens — they match via compounds
                    sw = jt[0]
                    if ref_pool.get(sw, 0) > 0:
                        ref_pool[sw] -= 1
                        if ref_pool[sw] == 0:
                            del ref_pool[sw]
                        jt[3] = True

                # Phase 2: Prefix fan-out — each prefix tries to compound with
                # ALL subsequent unmatched tokens (not just one).
                # e.g. "ne-" + "viens" → "neviens", "ne-" + "var" → "nevar"
                # The prefix token itself is marked matched if ≥1 compound works.
                # Each compounding root is also marked matched.
                for jti, jt in enumerate(json_token_list):
                    if jt[3]:
                        continue
                    ei = jt[2]
                    if ei not in prefix_raw_indices:
                        continue
                    prefix_stripped = jt[0]  # e.g. "ne"
                    any_compound_matched = False

                    for jtj in range(jti + 1, len(json_token_list)):
                        other = json_token_list[jtj]
                        if other[3]:  # already matched directly
                            continue
                        root_stripped = other[0]
                        compound = prefix_stripped + root_stripped
                        if ref_pool.get(compound, 0) > 0:
                            ref_pool[compound] -= 1
                            if ref_pool[compound] == 0:
                                del ref_pool[compound]
                            other[3] = True  # root consumed
                            any_compound_matched = True
                            # DON'T break — keep looking for more compounds

                    if any_compound_matched:
                        jt[3] = True  # prefix consumed

                # Phase 3: Suffix merging (root + suffix → compound)
                for jti, jt in enumerate(json_token_list):
                    if jt[3]:
                        continue
                    ei = jt[2]
                    if ei in prefix_raw_indices or ei in suffix_raw_indices:
                        continue
                    root_stripped = jt[0]
                    for jtj in range(jti + 1, len(json_token_list)):
                        other = json_token_list[jtj]
                        if other[3]:
                            continue
                        if other[2] not in suffix_raw_indices:
                            continue
                        compound = root_stripped + other[0]
                        if ref_pool.get(compound, 0) > 0:
                            ref_pool[compound] -= 1
                            if ref_pool[compound] == 0:
                                del ref_pool[compound]
                            jt[3] = True
                            other[3] = True
                            break

                # Phase 4: Try prefix tokens as direct matches if they
                # didn't compound with anything (ref might have standalone "ne")
                for jt in json_token_list:
                    if jt[3]:
                        continue
                    ei = jt[2]
                    if ei in prefix_raw_indices:
                        sw = jt[0]
                        if ref_pool.get(sw, 0) > 0:
                            ref_pool[sw] -= 1
                            if ref_pool[sw] == 0:
                                del ref_pool[sw]
                            jt[3] = True

                # --- 3e. Missing ref words → add to leftover_latvian ---
                new_leftovers = []
                for sw, count in ref_pool.items():
                    form = ref_stripped_to_form.get(sw, sw)
                    for _ in range(count):
                        new_leftovers.append(form)

                if new_leftovers:
                    if verbose:
                        print(f"  ➕🇱🇻 v{verse_num} adding leftovers: {new_leftovers}")
                    leftover_latvian.extend(new_leftovers)
                    fixes.append(('lv_leftover_add', key, new_leftovers))

                # --- 3f. Extra JSON tokens → remove ---
                unmatched_json = [jt for jt in json_token_list if not jt[3]]

                if unmatched_json:
                    excess_by_stripped = Counter()
                    for sw, orig, ei, matched in unmatched_json:
                        excess_by_stripped[sw] += 1

                    # Phase 1: Remove from leftover_latvian first
                    remaining_excess = Counter(excess_by_stripped)
                    new_leftover = []
                    removed_from_leftover = []
                    for item in leftover_latvian:
                        toks = lv_tokens(unicodedata.normalize('NFC', item))
                        if toks:
                            sw = strip_lv_diacritics(toks[0])
                            if remaining_excess.get(sw, 0) > 0:
                                remaining_excess[sw] -= 1
                                if remaining_excess[sw] == 0:
                                    del remaining_excess[sw]
                                removed_from_leftover.append(item)
                                continue
                        new_leftover.append(item)

                    if removed_from_leftover:
                        if verbose:
                            print(f"  ➖🇱🇻 v{verse_num} removed extra leftovers: {removed_from_leftover}")
                        leftover_latvian = new_leftover
                        fixes.append(('lv_leftover_trim', key, removed_from_leftover))

                    # Phase 2: Remove from greek_words[].latvian (walk backwards)
                    if remaining_excess:
                        removed_from_mapped = []
                        for gwi in reversed(range(len(greek_words))):
                            if not remaining_excess:
                                break
                            lv_list = greek_words[gwi].get('latvian', [])
                            new_lv = []
                            for item in reversed(lv_list):
                                toks = lv_tokens(unicodedata.normalize('NFC', item))
                                if toks:
                                    sw = strip_lv_diacritics(toks[0])
                                    if remaining_excess.get(sw, 0) > 0:
                                        remaining_excess[sw] -= 1
                                        if remaining_excess[sw] == 0:
                                            del remaining_excess[sw]
                                        removed_from_mapped.append(item)
                                        continue
                                new_lv.append(item)
                            new_lv.reverse()
                            if len(new_lv) != len(lv_list):
                                greek_words[gwi]['latvian'] = new_lv

                        if removed_from_mapped:
                            if verbose:
                                print(f"  ➖🇱🇻 v{verse_num} removed extra from mappings: {removed_from_mapped}")
                            fixes.append(('lv_mapped_trim', key, removed_from_mapped))

        # Write back into data
        data[vi]['greek_words'] = greek_words
        data[vi]['leftover_latvian'] = leftover_latvian

    # ── Write if any fixes ───────────────────────────────────────────────
    if fixes and not dry_run:
        write_dense_json(json_path, data)
        if verbose:
            print(f"  ✅ {len(fixes)} fix(es) written to {json_path}")

    return bool(fixes), fixes


## 4 – Discover & run on all chapters

In [5]:
def discover_json_chapters(base_dir):
    """Return dict: book_name -> sorted list of chapter numbers with .json files."""
    result = {}
    for book_dir in sorted(base_dir.iterdir()):
        if not book_dir.is_dir():
            continue
        chapters = sorted(
            int(f.stem) for f in book_dir.iterdir()
            if f.is_file() and f.suffix == '.json' and f.stem.isdigit()
        )
        if chapters:
            result[book_dir.name] = chapters
    return result

book_chapters = discover_json_chapters(BASE_DIR)
total_ch = sum(len(v) for v in book_chapters.values())
print(f"Found {len(book_chapters)} books, {total_ch} chapter JSONs")

Found 27 books, 260 chapter JSONs


In [6]:
# ── Run autofix on all chapters ──────────────────────────────────────────
all_fixes = []
files_changed = 0

for book_name, chapters in book_chapters.items():
    for ch_num in chapters:
        had_fixes, fix_log = autofix_chapter(
            book_name, ch_num, strongs_g, lv_g,
            verbose=VERBOSE, dry_run=DRY_RUN
        )
        if had_fixes:
            files_changed += 1
            all_fixes.extend(fix_log)

action = 'Would fix' if DRY_RUN else 'Fixed'
print(f"\n{'='*60}")
print(f"{action} {files_changed} file(s), {len(all_fixes)} total fix(es).")
if DRY_RUN:
    print("Set DRY_RUN = False and re-run to write changes.")


Fixed 260 file(s), 3642 total fix(es).


## 5 – Fix summary

In [7]:
fix_types = Counter(f[0] for f in all_fixes)
print("Fix breakdown:")
for t, c in fix_types.most_common():
    print(f"  {t}: {c}")

# Per-book summary
book_fix_counts = Counter()
for f in all_fixes:
    if len(f) >= 2 and isinstance(f[1], tuple) and len(f[1]) >= 1:
        book_fix_counts[f[1][0]] += 1

print(f"\nPer-book fix counts:")
for book, count in book_fix_counts.most_common():
    print(f"  {book}: {count}")

Fix breakdown:
  lv_mapped_trim: 2676
  lv_leftover_add: 965
  greek_clean: 1

Per-book fix counts:
  acts: 512
  luke: 495
  john: 458
  matthew: 352
  mark: 344
  1_corinthians: 236
  revelation: 218
  romans: 161
  2_corinthians: 137
  hebrews: 127
  galatians: 73
  ephesians: 71
  1_john: 65
  1_thessalonians: 53
  1_timothy: 50
  james: 44
  1_peter: 42
  colossians: 39
  2_peter: 32
  2_timothy: 29
  2_thessalonians: 26
  philippians: 26
  titus: 18
  jude: 14
  2_john: 10
  3_john: 7
  philemon: 3


## 6 – (Optional) Run on a single book/chapter for testing

In [8]:
# Test on a single chapter first
# had, log = autofix_chapter('acts', 11, strongs_g, lv_g, verbose=True, dry_run=True)
# print(f"\nFixes: {len(log)}")